# LangChain - Generating Dynamic Questions / SQL query pairs

**Goal**: to generate a set of questions and SQL pairs to ask about the Chinook database.

The questions and SQL queries should have dynamic fields, example json output:
``` json
{
    "question": "What is the total revenue generated by sales support agent <employee_last_name>?",
    "answer": "SELECT e.FirstName, e.LastName, SUM(i.Total) as total_revenue FROM Employee e JOIN Customer c ON e.EmployeeId = c.SupportRepId JOIN Invoice i ON c.CustomerId = i.CustomerId WHERE e.LastName = '<employee_last_name>';"
}
```


## Pydantic

Lang Chain uses Pydantic classes to know the JSON format in which to output results.

Below is a class for a question/SQL pair.

In [6]:
from pydantic import BaseModel, Field

class QuestionSQLPair(BaseModel):
    question: str = Field(description="A synthetic user question about the data.")
    sql_query: str = Field(description="A valid SQL query to answer the question.")

## Prompt Template

Fields:
- **template_fields**: A string in csv format containing the entire list of potential dynamic fields.
- **schema**: The complete schema of the database about which we need to generate questions.
- **target_table**: This field will allow me to break down the queriying to specific tables.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
        """
        You are a SQL expert.
        Your goal is to generate synthetic and dynamic User Questions and corresponding dynamic SQL queries based on a given data schema.
        
        **Rules:**
        1. Ensure the SQL is syntactically correct for SQLite.
        2. Use the provided Dynamic Fields placeholders (e.g. `<artist_name>`) instead of hardcoded values (e.g. 'AC/DC').
        3. Do not invent new dynamic fields; only use the ones listed below.

        **The "Spotlight" Rule:**
        You will be given a specific **Target Table**.
        Your questions MUST revolve around the metrics and attributes of that specific table.
        * **Good:** If Target is 'Employee', ask "Which Employee generated the most sales?" (Uses joins, but about Employee).
        * **Bad:** If Target is 'Employee', do not ask "Which Track is the longest?" (Irrelevant to Employee).

        **Available Dynamic Fields:**
```
{template_fields}
```
        
        **Few-Shot Examples:**
        
        Example 1: Time-Bound Sales
        {{
            "Q": "How many individual tracks did <artist_name> sell between <start_date> and <end_date>?",
            "A": "SELECT COUNT(il.InvoiceLineId) as total_tracks_sold FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId JOIN Album a ON t.AlbumId = a.AlbumId JOIN Artist art ON a.ArtistId = art.ArtistId JOIN Invoice i ON il.InvoiceId = i.InvoiceId WHERE art.Name = '<artist_name>' AND i.InvoiceDate BETWEEN '<start_date>' AND '<end_date>';"
        }}

        Example 2: Regional Market Analysis
        {{
            "Q": "List all customers in <country_name> who have purchased <genre_name> music.",
            "A": "SELECT DISTINCT c.FirstName, c.LastName, c.Email FROM Customer c JOIN Invoice i ON c.CustomerId = i.CustomerId JOIN InvoiceLine il ON i.InvoiceId = il.InvoiceId JOIN Track t ON il.TrackId = t.TrackId JOIN Genre g ON t.GenreId = g.GenreId WHERE g.Name = '<genre_name>' AND i.BillingCountry = '<country_name>';"
        }}

        Example 3: Employee Revenue
        {{
            "Q": "What is the total revenue generated by sales support agent <employee_last_name>?",
            "A": "SELECT e.FirstName, e.LastName, SUM(i.Total) as total_revenue FROM Employee e JOIN Customer c ON e.EmployeeId = c.SupportRepId JOIN Invoice i ON c.CustomerId = i.CustomerId WHERE e.LastName = '<employee_last_name>';"
        }}
        """
    ),
    ("human",
        """
        **Full Schema (Context):**
        {schema}

        **Target Table (Focus):**
        {target_table}

        **Task:**
        Generate 3 distinct questions specifically about the '{target_table}'.
        """
    )
])

# Print to ensure formatting is correct.
print(prompt.format(template_fields="artist_name, the full name of the artist ...", schema="CREATE TABLE...", target_table="Artist"))

System: 
        You are a SQL expert.
        Your goal is to generate synthetic and dynamic User Questions and corresponding dynamic SQL queries based on a given data schema.

        **Rules:**
        1. Ensure the SQL is syntactically correct for SQLite.
        2. Use the provided Dynamic Fields placeholders (e.g. `<artist_name>`) instead of hardcoded values (e.g. 'AC/DC').
        3. Do not invent new dynamic fields; only use the ones listed below.

        **The "Spotlight" Rule:**
        You will be given a specific **Target Table**.
        Your questions MUST revolve around the metrics and attributes of that specific table.
        * **Good:** If Target is 'Employee', ask "Which Employee generated the most sales?" (Uses joins, but about Employee).
        * **Bad:** If Target is 'Employee', do not ask "Which Track is the longest?" (Irrelevant to Employee).

        **Available Dynamic Fields:**
```
artist_name, the full name of the artist ...
```

        **Few-Shot Example

## Loading the Schema
I'm going to use the Chinook database in sqlite format for this test. I'm going to load and provide the entire schema because the LLM needs to be aware of all of the tables and the relationship beteween each other.

In [2]:
from functions import load_chinook_schema

ch_schema = load_chinook_schema(db_path="./data/Chinook_Sqlite.sqlite")

print(ch_schema)

CREATE TABLE [Album]
(
    [AlbumId] INTEGER  NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    CONSTRAINT [PK_Album] PRIMARY KEY  ([AlbumId]),
    FOREIGN KEY ([ArtistId]) REFERENCES [Artist] ([ArtistId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
)

CREATE TABLE [Artist]
(
    [ArtistId] INTEGER  NOT NULL,
    [Name] NVARCHAR(120),
    CONSTRAINT [PK_Artist] PRIMARY KEY  ([ArtistId])
)

CREATE TABLE [Customer]
(
    [CustomerId] INTEGER  NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    CONSTRAINT [PK_Customer] PRIMARY KEY  ([CustomerId]),
    FOREIGN KEY ([SupportRepId]) REFERENCES [Employee] ([EmployeeId]) 
		ON DELETE NO ACTION O